In [ ]:
# Importing the libraries
import numpy as np
import pandas as pd



from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import KFold

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [ ]:
# Importing dataset

from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

file_path = "/content/drive/MyDrive/student_productivity_distraction_dataset_20000.csv"

df = pd.read_csv(file_path)

df.head()
df.shape
df.info()
print(df.columns)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             20000 non-null  int64  
 1   age                    20000 non-null  int64  
 2   gender                 20000 non-null  object 
 3   study_hours_per_day    20000 non-null  float64
 4   sleep_hours            20000 non-null  float64
 5   phone_usage_hours      20000 non-null  float64
 6   social_media_hours     20000 non-null  float64
 7   youtube_hours          20000 non-null  float64
 8   gaming_hours           20000 non-null  float64
 9   breaks_per_day         20000 non-null  int64  
 10  coffee_intake_mg       20000 non-null  int64  
 11  exercise_minutes       20000 non-null  int64  
 12  assignments_completed  20000 

In [ ]:
# Process the dataset
df = df.drop(columns=[
    "student_id",
    "social_media_hours",
    "youtube_hours",
    "gaming_hours"
])
df = pd.get_dummies(df, columns=["gender"], drop_first=True)



# X = df.drop(columns=["phone_usage_hours", "productivity_score", "final_grade"])
X = df.drop("phone_usage_hours", axis=1)
y = df["phone_usage_hours"]

# add noise
np.random.seed(42)
X_noisy = X.copy()
numeric_features = X_noisy.select_dtypes(include=np.number).columns
X_noisy[numeric_features] += np.random.normal(0, 0.35, size=X_noisy[numeric_features].shape)
from sklearn.preprocessing import StandardScaler


from sklearn.model_selection import train_test_split
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_no = 1
mse_scores = []
mae_scores = []
r2_scores = []
gen_error = []


In [ ]:
# Build  model
for train_index, val_index in kf.split(X_noisy):

    X_train, X_val = X_noisy.iloc[train_index], X_noisy.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],),
                    kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dense(1)
    ])
    optimizer = keras.optimizers.Adam(learning_rate=0.0005)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

    # add early stopping
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )

    #train model
    print(f"Training Fold {fold_no}...")


    model.fit(
        X_train, y_train,
        epochs=200,
        batch_size=32,
        validation_data=(X_val, y_val),
        callbacks=[early_stop],
        verbose=0
    )
    y_pred = model.predict(X_val).flatten()
    y_train_pred = model.predict(X_train).flatten()

    train_mse = mean_squared_error(y_train, y_train_pred)
    val_mse = mean_squared_error(y_val, y_pred)

    generalization_error = val_mse - train_mse
    gen_error.append(generalization_error)

    mse_scores.append(mean_squared_error(y_val, y_pred))
    mae_scores.append(mean_absolute_error(y_val, y_pred))
    r2_scores.append(r2_score(y_val, y_pred))

    fold_no += 1




Training Fold 1...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Fold 2...
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


...

Training Fold 3...
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Fold 4...
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Fold 5...
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [ ]:

print("Multi-Layer Model Average MSE:", np.mean(mse_scores))
print("Multi-Layer Model MAE:", np.mean(mae_scores))
print("Multi-Layer Model R²:", np.mean(r2_scores))
print('Generalization Error:', np.mean(gen_error))

Multi-Layer Model Average MSE: 1.1166489960484771
Multi-Layer Model MAE: 0.8480363291330338
Multi-Layer Model R²: 0.8981573118206005
Generalization Error: 0.034888628833338896


In [ ]:
# Single hidden layer

fold_no = 1
mse_scores_simple = []
mae_scores_simple = []
r2_scores_simple = []
gen_error_simple = []

for train_index, val_index in kf.split(X_noisy):

    X_train, X_val = X_noisy.iloc[train_index], X_noisy.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    model = keras.Sequential([
        layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dropout(0.2),
        layers.Dense(1)
    ])

    optimizer = keras.optimizers.Adam(learning_rate=0.0005)

    model.compile(
        optimizer=optimizer,
        loss='mse',
        metrics=['mae']
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )

    print(f"Training Simple Model Fold {fold_no}...")

    model.fit(
        X_train, y_train,
        epochs=200,
        batch_size=32,
        validation_data=(X_val, y_val),
        callbacks=[early_stop],
        verbose=0
    )

    y_pred = model.predict(X_val).flatten()

    y_train_pred = model.predict(X_train).flatten()

    train_mse = mean_squared_error(y_train, y_train_pred)
    val_mse = mean_squared_error(y_val, y_pred)

    generalization_error = val_mse - train_mse
    gen_error_simple.append(generalization_error)

    mse_scores_simple.append(mean_squared_error(y_val, y_pred))
    mae_scores_simple.append(mean_absolute_error(y_val, y_pred))
    r2_scores_simple.append(r2_score(y_val, y_pred))

    fold_no += 1


print("Single Layer Model Average MSE:", np.mean(mse_scores_simple))
print("Single Layer Model Average MAE:", np.mean(mae_scores_simple))
print("Single Layer Average R²:", np.mean(r2_scores_simple))
print('Generalization Error:', np.mean(gen_error_simple))

Training Simple Model Fold 1...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Simple Model Fold 2...
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


...

Training Simple Model Fold 3...
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Simple Model Fold 4...
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Simple Model Fold 5...
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Single Layer Model Average MSE: 1.1018272253762955
Single Layer Model Average MAE: 0.839473322576046
Single Layer Average R²: 0.8995240325557141
Generalization Error: 0.01309354184523821


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/student_productivity_distraction_dataset_20000.csv"
df = pd.read_csv(file_path)


df = df.drop(columns=["student_id"])
df = pd.get_dummies(df, columns=["gender"], drop_first=True)

correlations = df.corr()['phone_usage_hours'].sort_values(ascending=False)

print("Correlation of all features with phone_usage_hours:\n")
print(correlations)

Correlation of all features with phone_usage_hours:

phone_usage_hours        1.000000
study_hours_per_day      0.011539
age                      0.009090
assignments_completed    0.006213
stress_level             0.006138
youtube_hours            0.004108
social_media_hours       0.004030
sleep_hours              0.000016
coffee_intake_mg        -0.000064
focus_score             -0.000206
gender_Other            -0.002913
attendance_percentage   -0.002986
exercise_minutes        -0.003186
gender_Male             -0.005174
breaks_per_day          -0.005431
gaming_hours            -0.007497
final_grade             -0.012136
productivity_score      -0.326650
Name: phone_usage_hours, dtype: float64
